**Features Extraction**

In [1]:
import os
import librosa
import pandas as pd
import numpy as np

In [8]:
def extract_features(ef_data: np.ndarray, ef_sample_rate: int) -> np.ndarray:

    ef_features = np.array([])

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, zcr))

    # Chroma STFT
    stft = librosa.stft(ef_data)
    chroma_stft = np.mean(librosa.feature.chroma_stft(S=np.abs(stft), sr=ef_sample_rate).T, axis=0)
    ef_features = np.hstack((ef_features, chroma_stft))

    # MFCC
    mfcc = np.mean(librosa.feature.mfcc(y=ef_data, sr=ef_sample_rate).T, axis=0)
    ef_features = np.hstack((ef_features, mfcc))

    # Root Mean Square Value
    rmse = np.mean(librosa.feature.rms(y=ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, rmse))

    # Mel Spectrogram
    mel = np.mean(librosa.feature.melspectrogram(y=ef_data, sr=ef_sample_rate).T, axis=0)
    ef_features = np.hstack((ef_features, mel))

    return ef_features

def get_features(gf_path: str) -> np.ndarray:
    data, sample_rate = librosa.load(gf_path, sr=None)

    sample_rate = int(sample_rate)
    features = extract_features(data, sample_rate)

    return features

def prepare_audios(pa_df: pd.DataFrame, pa_name: str):
    data = []
    labels = []
    total = len(pa_df)

    for _, row in pa_df.iterrows():
        try:
            features = get_features(row['path'])
            data.append(features)
            labels.append(row['emotion'])
            print(f"{pa_name} - Done! Saved {len(data)}/{total}", end="\r", flush=True)

        except Exception as e:
            print(f"Error processing {row['path']}: {e}")

    data = np.array(data)
    labels = np.array(labels)

    if not os.path.exists("features"):
        os.makedirs("features")

    np.save(f"features\\{pa_name}_data.npy", data)
    print()
    np.save(f"features\\{pa_name}_labels.npy", labels)

data_path_Train = pd.read_csv("CSVs\\ravdess_train.csv")
data_path_Test = pd.read_csv("CSVs\\ravdess_test.csv")

prepare_audios(data_path_Train, "train")
prepare_audios(data_path_Test, "test")

train - Done! Saved 1152/1152
test - Done! Saved 288/288
